In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/eya-bouhmida/Multimodal-RAG-From-Scratch.git

import os, shutil
os.chdir('/content/Multimodal-RAG-From-Scratch')

if os.path.exists('data'):
    if os.path.islink('data'):
        os.unlink('data')
    else:
        shutil.rmtree('data')

os.symlink(
    '/content/drive/MyDrive/multimodal-rag-project/data',
    '/content/Multimodal-RAG-From-Scratch/data'
)
print(os.listdir('data/raw/'))

In [ ]:
!pip install sentence-transformers qdrant-client rank-bm25 groq -q

In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
from rank_bm25 import BM25Okapi
from groq import Groq

In [ ]:
!pip install python-dotenv -q

from dotenv import load_dotenv
import os

load_dotenv()

QDRANT_URL = "https://a52409e3-d81f-4182-a9f5-a23b1511daef.australia-southeast1-0.gcp.cloud.qdrant.io"
COLLECTION_NAME = "medlens"
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

groq_client = Groq(api_key=GROQ_API_KEY)

print(f"✅ Connecté!")
print(f"Text chunks: {client.count('medlens').count}")
print(f"Image chunks: {client.count('medlens_images').count}")

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f"✅ Modèles chargés sur {device}!")

In [ ]:
print("⏳ Loading BM25...")

all_chunks = []
results, offset = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=5000,
    offset=None,
    with_payload=True,
    with_vectors=False
)
all_chunks.extend(results)

tokenized_chunks = [chunk.payload['text'].lower().split() for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"✅ BM25 ready with {len(all_chunks)} chunks!")

In [ ]:
def dense_search(query, top_k=20):
    query_vector = embedding_model.encode(query).tolist()
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True
    ).points
    return results

def bm25_search(query, top_k=20):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{"id": all_chunks[idx].id, "score": scores[idx], "payload": all_chunks[idx].payload} for idx in top_indices]

def reciprocal_rank_fusion(dense_results, bm25_results, k=60):
    scores = {}
    for rank, result in enumerate(dense_results):
        doc_id = result.id
        if doc_id not in scores:
            scores[doc_id] = {"score": 0, "payload": result.payload}
        scores[doc_id]["score"] += 1 / (k + rank + 1)
    for rank, result in enumerate(bm25_results):
        doc_id = result["id"]
        if doc_id not in scores:
            scores[doc_id] = {"score": 0, "payload": result["payload"]}
        scores[doc_id]["score"] += 1 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True)[:20]

def rerank(query, fused_results, top_k=8):
    pairs = [[query, result[1]["payload"]["text"]] for result in fused_results]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, fused_results), key=lambda x: x[0], reverse=True)
    return ranked[:top_k]

def hybrid_search(query, top_k=8):
    dense_results = dense_search(query, top_k=20)
    bm25_results = bm25_search(query, top_k=20)
    fused = reciprocal_rank_fusion(dense_results, bm25_results)
    return rerank(query, fused, top_k=top_k)

In [ ]:
def image_search(query, top_k=3):
    query_vector = embedding_model.encode(query).tolist()
    results = client.query_points(
        collection_name="medlens_images",
        query=query_vector,
        limit=top_k,
        with_payload=True
    ).points
    return results

In [ ]:
def build_multimodal_prompt(query, text_chunks, image_results):
    context = ""
    sources = []

    context += "=== MEDICAL DOCUMENTS ===\n"
    for i, (score, result) in enumerate(text_chunks):
        payload = result[1]["payload"]
        context += f"\n--- Document {i+1} ---\n"
        context += f"Source: {payload['filename']} (Page {payload['page_num']})\n"
        context += f"{payload['text']}\n"
        sources.append(f"{payload['filename']} p.{payload['page_num']}")

    if image_results:
        context += "\n=== MEDICAL FIGURES AND IMAGES ===\n"
        for i, img in enumerate(image_results):
            context += f"\n--- Figure {i+1} ---\n"
            context += f"Image: {img.payload['filename']}\n"
            context += f"Description: {img.payload['text']}\n"
            sources.append(f"Figure: {img.payload['filename']}")

    prompt = f"""You are MedLens, an expert bilingual medical assistant (French/English).

ABSOLUTE RULES:
1. Answer ONLY using information from the documents provided below.
2. Do NOT use your own general knowledge under any circumstances.
3. Every statement must be directly traceable to the provided documents.
4. If the information is not in the documents, say exactly: "I cannot find this information in the available documents."
5. After each important statement, cite the source in parentheses (filename and page number).
6. Always respond in the same language as the question.
7. Be precise, clear, and helpful — you are helping non-medical people understand health information.

MEDICAL CONTEXT:
{context}

QUESTION: {query}

DETAILED CITED RESPONSE:"""

    return prompt, sources

In [ ]:
def generate_response(query):
    # Retrieval with top_k=8
    text_chunks = hybrid_search(query, top_k=8)
    image_results = image_search(query, top_k=3)
    prompt, sources = build_multimodal_prompt(query, text_chunks, image_results)

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1200,
        temperature=0.05  # ✅ lower temperature = more faithful
    )

    answer = response.choices[0].message.content

    return {
        "question": query,
        "answer": answer,
        "sources": sources,
        "text_chunks": len(text_chunks),
        "images_used": len(image_results)
    }

In [ ]:
from IPython.display import display, Image as IPImage

def generate_response_with_images(query):
    text_chunks = hybrid_search(query, top_k=8)
    image_results = image_search(query, top_k=3)
    prompt, sources = build_multimodal_prompt(query, text_chunks, image_results)

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1200,
        temperature=0.05
    )

    answer = response.choices[0].message.content

    print(f"❓ Question: {query}")
    print(f"\n🤖 MedLens:\n{answer}")
    print(f"\n📄 Sources: {sources[:5]}")

    if image_results:
        print(f"\n🖼️ Relevant medical images ({len(image_results)}):")
        for i, img in enumerate(image_results):
            img_path = img.payload.get('image_path', '')
            caption = img.payload.get('text', '')
            print(f"\n--- Figure {i+1} ---")
            print(f"📝 {caption[:150]}...")
            if os.path.exists(img_path):
                display(IPImage(filename=img_path, width=400))
            else:
                print(f"⚠️ Image not found locally")

In [ ]:
generate_response_with_images("What are the symptoms of diabetes?")

In [ ]:
generate_response_with_images("Quels sont les traitements pour l'hypertension?")

In [ ]:
result = generate_response("What is the recommended treatment for type 2 diabetes?")
print(f"✅ Answer quality check:")
print(f"Text chunks used: {result['text_chunks']}")
print(f"Images used: {result['images_used']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nSources: {result['sources']}")